## Init

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql.functions import row_number, lit

import sys
import os

current_directory = os.getcwd()
if current_directory not in sys.path:
    sys.path.append(current_directory)

from script.utils.config import cities_config

## Create DataFrame from static dictionary

In [0]:
df = spark.createDataFrame(cities_config)

windowSpec = Window.orderBy(lit('state'))
df_state_code = (
    df
    .withColumn(
        "state_code", 
        row_number().over(windowSpec)
    )
)

df_cities = (
    df_state_code
    .select(
        "state_code", "state", "city", "lat", "lon"
    )
)

## Write data in silver table 
Using MERGE with (state_name + city_name) as unique key 

In [ ]:
table_name = "weather.cities"

if not spark.catalog.tableExists(table_name):
    (
        df_cities
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
else: 
    target_table = DeltaTable.forName(spark, table_name)

    (
        target_table.alias("target")
        .merge(
            df_cities.alias("source"), 
            condition="""
            target.state = source.state AND 
            target.city = source.city
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

## Sanity check

In [0]:
%sql
select * 
from weather.cities
order by state_code
limit 10